<a href="https://colab.research.google.com/github/kzettl/TTAI2800StudentWork/blob/main/Lab_6_Complete.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Imports and reproducibility
import numpy as np, pandas as pd
from pathlib import Path

rng = np.random.default_rng(42)
out = Path('.')

## Define our simple event schema
We keep this small and friendly: categorical fields (role, resource_type, etc.) and a few numeric fields we can plot and test for drift.

In [4]:
users = [f'user_{i}' for i in range(530)]
roles = ['engineer','analyst','service','admin']
resource_types = ['code_repo','db_table','secrets_vault','build_system','analytics']
actions = ['read','write','delete','configure']
geo = ['NA','EU','APAC','LATAM']

def sample_base(n):
    return pd.DataFrame({
        'timestamp': rng.integers(1_700_000_000, 1_700_086_400, size=n),
        'user_id': rng.choice(users, size=n),
        'role': rng.choice(roles, size=n, p=[0.45,0.30,0.20,0.05]),
        'resource_type': rng.choice(resource_types, size=n),
        'action': rng.choice(actions, size=n, p=[0.55,0.25,0.05,0.15]),
        'bytes_transferred': np.round(rng.lognormal(mean=10, sigma=1, size=n)).astype(int),
        'hour_of_day': rng.integers(0,24,size=n),
        'geo_region': rng.choice(geo, size=n, p=[0.55,0.20,0.15,0.10]),
        'ip_reputation_score': rng.beta(8,2,size=n),
        'session_length_sec': np.round(rng.gamma(4, 120, size=n)).astype(int)
    })

## Generate baseline and inject a few anomaly patterns
We'll create:
- Burst outliers (huge bytes on secrets_vault writes)
- Backdoor trigger pattern (rare combo repeated)
- Slow drift (ip reputation distribution shifts for subset of users)
- Uniform noise block (simulated corrupted feed)

In [5]:
N = 9000
base = sample_base(N)
base['anomaly_type'] = 'base'

# Burst outliers
burst = sample_base(120)
burst['bytes_transferred'] *= rng.integers(50,120,size=len(burst))
burst['resource_type'] = 'secrets_vault'
burst['action'] = 'write'
burst['anomaly_type'] = 'burst'

# Backdoor trigger
backdoor = sample_base(150)
backdoor['role'] = 'admin'
backdoor['resource_type'] = 'code_repo'
backdoor['action'] = 'configure'
backdoor['hour_of_day'] = 3
backdoor['anomaly_type'] = 'backdoor'

# Slow drift (ip_reputation_score becomes more uniform for subset)
drift = sample_base(180)
mask = drift['user_id'].isin(drift['user_id'].unique()[:40])
drift.loc[mask,'ip_reputation_score'] = rng.beta(2,2,size=mask.sum())
drift['anomaly_type'] = 'drift'

# Uniform noise block
noise = sample_base(100)
for col in ['bytes_transferred','ip_reputation_score','session_length_sec']:
    noise[col] = rng.uniform(low=0, high=max(1, noise[col].max())*1.2, size=len(noise))
noise['anomaly_type'] = 'noise'

dataset = pd.concat([base, burst, backdoor, drift, noise], ignore_index=True)
dataset = dataset.sample(frac=1.0, random_state=42).reset_index(drop=True)  # shuffle

## Save outputs
We save the full mixed dataset and also a baseline-only snapshot for training an unsupervised model later.

In [6]:
dataset.to_csv(out / 'events_raw.csv', index=False)
base.to_csv(out / 'events_baseline.csv', index=False)
print('Saved:', (out/'events_raw.csv').resolve())
print('Saved:', (out/'events_baseline.csv').resolve())
dataset['anomaly_type'].value_counts()

Saved: /content/events_raw.csv
Saved: /content/events_baseline.csv


,count
anomaly_type,
base,9000
drift,180
backdoor,150
burst,120
noise,100


In [7]:
import pandas as pd, hashlib, json
from pathlib import Path

root = Path('.')
df = pd.read_csv(root / 'events_raw.csv')
print('Loaded rows:', len(df))
df.head(3)

Loaded rows: 9550


,timestamp,user_id,role,resource_type,action,bytes_transferred,hour_of_day,geo_region,ip_reputation_score,session_length_sec,anomaly_type
0,1700059202,user_187,engineer,analytics,read,49740.0,0,NaN,0.929821,171.0,base
1,1700067177,user_403,analyst,analytics,write,3277.0,20,APAC,0.890900,313.0,base
2,1700073608,user_204,engineer,secrets_vault,write,11528535.0,1,NaN,0.940853,456.0,burst


In [8]:
required_cols = [
  'timestamp','user_id','role','resource_type','action','bytes_transferred',
  'hour_of_day','geo_region','ip_reputation_score','session_length_sec','anomaly_type'
]
missing = [c for c in required_cols if c not in df.columns]
assert not missing, f'Missing columns: {missing}'

assert df['hour_of_day'].between(0,23).all(), 'hour_of_day out of range'
assert (df['bytes_transferred']>=0).all(), 'negative bytes'
print('Basic schema checks passed')

Basic schema checks passed


In [9]:
def sha256(path):
    h = hashlib.sha256()
    with open(path,'rb') as f: h.update(f.read())
    return h.hexdigest()

manifest = {
  'rows': int(len(df)),
  'freq_role': df['role'].value_counts(normalize=True).to_dict(),
  'freq_resource_type': df['resource_type'].value_counts(normalize=True).to_dict(),
  'numeric_summary': df[['bytes_transferred','ip_reputation_score','session_length_sec']].describe().to_dict(),
  'file': 'events_raw.csv',
  'sha256': sha256(root/'events_raw.csv')
}
with open(root/'manifest_raw.json','w') as f: json.dump(manifest,f,indent=2)
print('Wrote manifest_raw.json')
manifest['sha256'][:16] + '…'


Wrote manifest_raw.json


'aa5938184d65120d…'

In [10]:


import numpy as np, pandas as pd
from scipy import stats
from pathlib import Path

root = Path('.')
base = pd.read_csv(root/'events_baseline.csv')
current = pd.read_csv(root/'events_raw.csv')
print(len(base), len(current))
base.head(2)



9000 9550


,timestamp,user_id,role,resource_type,action,bytes_transferred,hour_of_day,geo_region,ip_reputation_score,session_length_sec,anomaly_type
0,1700007711,user_236,service,secrets_vault,read,17236,21,NaN,0.791438,407,base
1,1700066869,user_9,engineer,code_repo,read,56856,17,EU,0.894517,886,base


In [11]:

def psi(expected, actual, bins=10):
    e_hist, edges = np.histogram(expected, bins=bins)
    a_hist, _ = np.histogram(actual, bins=edges)
    e_ratio = np.clip(e_hist / max(1,len(expected)), 1e-6, None)
    a_ratio = np.clip(a_hist / max(1,len(actual)), 1e-6, None)
    return float(np.sum((a_ratio - e_ratio) * np.log(a_ratio / e_ratio)))

numeric = ['bytes_transferred','ip_reputation_score','session_length_sec']
psi_scores = {c: psi(base[c].values, current[c].values) for c in numeric}
ks_pvals = {c: float(stats.ks_2samp(base[c].values, current[c].values).pvalue) for c in numeric}
psi_scores, ks_pvals



({'bytes_transferred': 0.012625955918488774,
  'ip_reputation_score': 0.0032256655594371225,
  'session_length_sec': 0.00021911541080295573},
 {'bytes_transferred': 0.13282233753400904,
  'ip_reputation_score': 0.9709790927668253,
  'session_length_sec': 0.9999999618484495})

In [12]:
import json
drifted = []
for c in numeric:
    if psi_scores[c] > 0.25 or ks_pvals[c] < 0.01:
        drifted.append(c)
report = { 'psi': psi_scores, 'ks_pvalue': ks_pvals, 'drifted_features': drifted }
with open(root/'drift_report.json','w') as f: json.dump(report,f,indent=2)
report


{'psi': {'bytes_transferred': 0.012625955918488774,
  'ip_reputation_score': 0.0032256655594371225,
  'session_length_sec': 0.00021911541080295573},
 'ks_pvalue': {'bytes_transferred': 0.13282233753400904,
  'ip_reputation_score': 0.9709790927668253,
  'session_length_sec': 0.9999999618484495},
 'drifted_features': []}

In [13]:

import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import joblib
from pathlib import Path

root = Path('.')
base = pd.read_csv(root/'events_baseline.csv')
data = pd.read_csv(root/'events_raw.csv')
len(base), len(data)

(9000, 9550)

In [14]:

numeric = ['bytes_transferred','hour_of_day','ip_reputation_score','session_length_sec']
categorical = ['role','resource_type','action','geo_region']
ct = ColumnTransformer([
    ('num', StandardScaler(), numeric),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical)
])
iso = IsolationForest(n_estimators=200, contamination=0.03, random_state=42)
pipe = Pipeline([('prep', ct), ('iso', iso)])
pipe.fit(base[numeric+categorical])
joblib.dump(pipe, root/'iforest_model.joblib')
print('Model saved to iforest_model.joblib')

Model saved to iforest_model.joblib


In [15]:
pipe = joblib.load(root/'iforest_model.joblib')
X = data[numeric+categorical]
scores = pipe['iso'].score_samples(pipe['prep'].transform(X))
data['iso_score'] = scores
data.head(3)

,timestamp,user_id,role,resource_type,action,bytes_transferred,hour_of_day,geo_region,ip_reputation_score,session_length_sec,anomaly_type,iso_score
0,1700059202,user_187,engineer,analytics,read,49740.0,0,NaN,0.929821,171.0,base,-0.461004
1,1700067177,user_403,analyst,analytics,write,3277.0,20,APAC,0.890900,313.0,base,-0.523938
2,1700073608,user_204,engineer,secrets_vault,write,11528535.0,1,NaN,0.940853,456.0,burst,-0.536924


In [16]:
data.to_csv(root/'events_scored.csv', index=False)
print('Saved events_scored.csv with iso_score column')

Saved events_scored.csv with iso_score column


In [17]:

import pandas as pd, numpy as np, json
from pathlib import Path

root = Path('.')
df = pd.read_csv(root/'events_scored.csv')
len(df), df.head(2)

(9550,
     timestamp   user_id      role resource_type action  bytes_transferred  \
 0  1700059202  user_187  engineer     analytics   read            49740.0   
 1  1700067177  user_403   analyst     analytics  write             3277.0   
 
    hour_of_day geo_region  ip_reputation_score  session_length_sec  \
 0            0        NaN             0.929821               171.0   
 1           20       APAC             0.890900               313.0   
 
   anomaly_type  iso_score  
 0         base  -0.461004  
 1         base  -0.523938  )

In [18]:
threshold = float(np.quantile(df['iso_score'].values, 0.03))
df['iso_anomaly'] = df['iso_score'] < threshold
threshold, df['iso_anomaly'].mean()

(-0.5806207092928029, np.float64(0.03005235602094241))

In [19]:
y_true = df['anomaly_type'].ne('base')  # True if injected
y_pred = df['iso_anomaly']
tp = int(((y_true==1)&(y_pred==1)).sum())
fp = int(((y_true==0)&(y_pred==1)).sum())
fn = int(((y_true==1)&(y_pred==0)).sum())
precision = tp / max(1, tp+fp)
recall = tp / max(1, tp+fn)
f1 = 2*precision*recall/max(1e-9, precision+recall)
metrics = {'precision': float(precision), 'recall': float(recall), 'f1': float(f1), 'threshold': threshold}
with open(root/'report_metrics.json','w') as f: json.dump(metrics,f,indent=2)
metrics

{'precision': 0.43205574912891986,
 'recall': 0.22545454545454546,
 'f1': 0.29629629629629634,
 'threshold': -0.5806207092928029}